# Validasi MuQ saja dengan encoder Core ML (tanpa MusicFM)

Buka dari root **SongMap**; memakai Python `.venv-songformer311` di sibling Section-Annotations.

Tahap sebelumnya memakai fitur MuQ dari cache PyTorch. Di sini setiap lagu validation (32) di-decode dari MP3, lalu fitur MuQ dihitung ulang dengan model yang akan dipakai di iPhone: frontend log-mel dan konvolusi MuQ dari app, serta **bank MuQ canonical palette6 yang sudah ada** (bobot tidak berubah). Setelah itu head palette6 dan konteks Core ML MuQ. MusicFM tidak dimuat sama sekali. Semua di CPU Mac.

Gerbang (tetap sebelum hasil): akurasi dan macro F1 dengan encoder Core ML masing-masing turun maks. 0,01 dibanding konfigurasi yang sama pada fitur cache asli (0,8049 / 0,7692 dari verifikasi sebelumnya), pada section yang identik.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/muq_encoder_quality.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
# Hanya folder ekspor yang lengkap; file log dengan awalan yang sama diabaikan.
EXPORT=sorted(p for p in (ROOT/'outputs.local').glob('muq-coreml-export-*') if (p/'export_report.json').is_file())[-1]
VERIFICATION=sorted((ROOT/'outputs.local').glob('muq-coreml-verify-*/verification.json'))[-1]
assert json.loads(VERIFICATION.read_text())['passed'], 'Verifikasi export belum lolos'
# Bank MuQ canonical yang sudah lolos validasi penuh sebelumnya (report kualitas yang ditinjau).
QUALITY=next(p for p in (ROOT/'outputs.local').glob('encoder-canonical-bank-*/quality/validation-all.json')
             if sha(p)=='020f8d9a2bb5c29318f591a7c58d30b8f4cdc39eba47cf82c09820b538ac96fa')
wanted=json.loads((QUALITY.parent/'selection.json').read_text())['reports']['muq']
MUQ_REPORT=next(p for p in QUALITY.parent.parent.rglob('*.json') if p.stat().st_size<5_000_000 and sha(p)==wanted)
OUTPUT=ROOT/'outputs.local'/('muq-encoder-quality-'+EXPORT.name.removeprefix('muq-coreml-export-'))
print('Export:',EXPORT); print('Verifikasi:',VERIFICATION); print('Bank MuQ:',MUQ_REPORT); print('Output:',OUTPUT)

In [ ]:
def run(limit):
    log=OUTPUT.parent/(OUTPUT.name+f'-limit{limit}.log')
    command=[str(RUN_PYTHON),'-u','scripts/muq_encoder_quality.py','--songstruct-root',str(SONGSTRUCT_ROOT),
        '--songmap-root',str(ROOT),'--export-dir',str(EXPORT),'--verification',str(VERIFICATION),
        '--muq-report',str(MUQ_REPORT),'--output-dir',str(OUTPUT),'--limit',str(limit)]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status: raise RuntimeError(f'Gagal ({status}); lihat {log}')

run(1)   # uji cepat satu lagu; bukan keputusan

In [ ]:
run(0)   # 32 lagu; lagu yang sudah selesai dipakai ulang
r=json.loads((OUTPUT/'validation-all.json').read_text())
print(json.dumps({k:r[k] for k in ('reference_original_features','candidate_coreml_encoder','encoder_function_calls','passed')},indent=2))
print('Bagikan:',OUTPUT/'validation-all.json')

Bila lolos, tahap berikutnya adalah probe iPhone tanpa MusicFM: ukuran terpasang, waktu analisis dan memori dibandingkan pipeline canonical dua-encoder sekarang (±44 detik Alleluia, ±117 detik Raise a Hallelujah).